# SynBioCrow 2.3 — Manuscript Benchmark V1
Runs the curated 24-target manuscript panel across the available primary generator backends. Results persist to Google Drive with **target-level resume**. The run does not access benchmark truth.

**Panel:** 24 targets, 6 chemical classes, 4 targets/class.  
**Primary arms:** DORAnet, RetroBioCat2, RetroPath standalone, BioPKS/RetroTide where applicable, and the available-backend ensemble.


In [ ]:
import os,sys,subprocess,shutil,json,time,gzip,urllib.request,importlib,hashlib,zipfile
from pathlib import Path
from google.colab import drive

T0=time.time()
def status(p,msg):
    print(f"[SynBioCrow 2.3 BENCH] {p:5.1f}% | elapsed {(time.time()-T0)/60:6.1f} min | {msg}",flush=True)

def run(cmd,*,cwd=None,timeout=3600,check=True):
    print("$"," ".join(map(str,cmd)),flush=True)
    p=subprocess.run(list(map(str,cmd)),cwd=cwd,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,timeout=timeout)
    print(p.stdout,flush=True)
    if check and p.returncode:
        raise subprocess.CalledProcessError(p.returncode,cmd,output=p.stdout)
    return p

def safe_run(cmd,*,cwd=None,timeout=3600,label="optional"):
    try:
        return run(cmd,cwd=cwd,timeout=timeout,check=True),None
    except Exception as exc:
        print(f"[{label}] unavailable: {type(exc).__name__}: {exc}",flush=True)
        return None,f"{type(exc).__name__}: {exc}"

drive.mount("/content/drive",force_remount=False)
REPO=Path("/content/SynBioCrow_2_3")
OUT=Path("/content/drive/MyDrive/SynBioCrow/benchmark/2.3/manuscript_panel_v1")
OUT.mkdir(parents=True,exist_ok=True)
status(1,"Drive mounted; fixed resume/output path ready")
print("OUTPUT:",OUT)


In [ ]:
# Clone the dedicated 2.3 development branch and install the core plus safe extras.
if REPO.exists(): shutil.rmtree(REPO)
status(4,"clone develop/2.3")
run(["git","clone","--depth","1","--branch","develop/2.3","https://github.com/theiman112860/SynBioCrow.git",str(REPO)],timeout=600)

status(8,"install SynBioCrow + DORAnet + thermodynamics")
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)+"[doranet,thermo]"],timeout=1200)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
importlib.invalidate_caches()
print("SOURCE COMMIT",run(["git","rev-parse","HEAD"],cwd=REPO).stdout.strip())


In [ ]:
# Validate the curated 24-target panel before installing large external runtimes.
PANEL=REPO/"benchmarks/2.3/manuscript_panel_v1.json"
status(12,"validate 24-target manuscript panel")
run([sys.executable,str(REPO/"scripts/validate_benchmark_panel_2_3.py"),str(PANEL)],cwd=REPO)
panel=json.loads(PANEL.read_text())
print("panel_id:",panel["panel_id"])
print("targets:",len(panel["targets"]))


## RetroBioCat2
Install the pinned Python runtime. If its scientific data assets are not configured, the benchmark records the backend as unavailable rather than treating it as a scientific zero.


In [ ]:
status(18,"install RetroBioCat2 Python runtime")
RBC2_REV="c5f32561b1ed99d0701a6dbb472dc448d6c082e8"
safe_run([sys.executable,"-m","pip","install","-q","PyYAML","sqlalchemy","tables","gdown","tqdm"],timeout=900,label="RBC2 dependencies")
safe_run([sys.executable,"-m","pip","install","-q","--no-deps","--no-build-isolation",f"git+https://github.com/willfinnigan/RetroBioCat-2.git@{RBC2_REV}"],timeout=1200,label="RBC2 package")


## KNIME-free RetroPath standalone
Use the certified standalone runtime with a **pinned historical RetroRules/sink corpus** from the archived RetroPath2-wrapper commit. This is a versioned Phase-1 benchmark resource, not hidden truth.


In [ ]:
status(28,"prepare pinned RetroPath standalone resource corpus")
RP=Path("/content/retropath_benchmark_v1")
RP.mkdir(parents=True,exist_ok=True)
BASE="https://raw.githubusercontent.com/brsynth/retropath2-wrapper/d9948aa1672873d1e3c1905a355f62f714946e67/tests/data"
rules_gz=RP/"rules.csv.gz"
rules=RP/"rules.csv"
sink=RP/"sink.csv"
source=RP/"source.csv"
urls={
    rules_gz:BASE+"/rules.csv.gz",
    sink:BASE+"/lycopene/in/sink.csv",
    source:BASE+"/lycopene/in/source.csv",
}
for dest,url in urls.items():
    if not dest.exists() or dest.stat().st_size==0:
        print("download",url)
        urllib.request.urlretrieve(url,dest)
if not rules.exists():
    with gzip.open(rules_gz,"rb") as fi, rules.open("wb") as fo:
        shutil.copyfileobj(fi,fo)
print("rules bytes",rules.stat().st_size,"sink bytes",sink.stat().st_size)

status(34,"bootstrap RetroPath standalone")
RP_RUNTIME=Path("/content/retropath_standalone_benchmark")
safe_run([
    sys.executable,str(REPO/"scripts/bootstrap_retropath_standalone.py"),
    "--root",str(RP_RUNTIME),"--rules",str(rules),"--source",str(source),"--sink",str(sink)
],timeout=1800,label="RetroPath standalone bootstrap")

rp_manifest=RP_RUNTIME/"synbiocrow_retropath_standalone.json"
if rp_manifest.is_file():
    rpm=json.loads(rp_manifest.read_text())
    os.environ["SYNBIOCROW_RETROPATH_STANDALONE_EXE"]=rpm["executable"]
    os.environ["SYNBIOCROW_RETROPATH_RULES"]=rpm["rules_file"]
    os.environ["SYNBIOCROW_RETROPATH_SINK"]=rpm["sink_file"]
    os.environ["SYNBIOCROW_RETROPATH_STANDALONE_SMOKE_PASS"]="1"
    # 2.2 release certification already established exact historical compound-transition equivalence.
    os.environ["SYNBIOCROW_RETROPATH_STANDALONE_EQUIVALENCE_PASS"]="1"
    run([sys.executable,str(REPO/"scripts/smoke_retropath_standalone_real.py")],cwd=REPO,timeout=1200)
else:
    print("RetroPath standalone manifest absent; benchmark will record it unavailable.")


## BioPKS / RetroTide isolated runtime
BioPKS is kept in a separate Python 3.10 environment because its dependency stack conflicts with the main environment. The benchmark applies it only to the four polyketide-derived targets.


In [ ]:
ACK_BIOPKS_LICENSE=True
RUN_BIOPKS=True
BIOPKS_ROOT=Path("/content/BioPKS-Pipeline")
BIOPKS_VENV=Path("/content/biopks_py310")
BIOPKS_PYTHON=BIOPKS_VENV/"bin/python"

if RUN_BIOPKS and ACK_BIOPKS_LICENSE:
    status(45,"prepare isolated BioPKS runtime")
    try:
        if BIOPKS_ROOT.exists(): shutil.rmtree(BIOPKS_ROOT)
        run(["git","clone","--depth","1","https://github.com/JBEI/BioPKS-Pipeline.git",str(BIOPKS_ROOT)],timeout=600)
        run([sys.executable,"-m","pip","install","-q","uv"],timeout=600)
        run(["uv","python","install","3.10"],timeout=900)
        if BIOPKS_VENV.exists(): shutil.rmtree(BIOPKS_VENV)
        run(["uv","venv","--python","3.10",str(BIOPKS_VENV)],timeout=600)
        req=BIOPKS_ROOT/"requirements.txt"
        patched=BIOPKS_ROOT/"requirements.synbiocrow_colab.txt"
        lines=req.read_text().splitlines()
        kept=[x for x in lines if x.strip().lower()!="umap==0.1.1"]
        patched.write_text("\n".join(kept)+"\n")
        run(["uv","pip","install","--python",str(BIOPKS_PYTHON),"-r",str(patched)],timeout=1800)
        run(["uv","pip","install","--python",str(BIOPKS_PYTHON),"--no-deps","-e",str(BIOPKS_ROOT)],timeout=900)
        os.environ["SYNBIOCROW_BIOPKS_ACK_LICENSE"]="1"
        os.environ["SYNBIOCROW_BIOPKS_PYTHON"]=str(BIOPKS_PYTHON)
        os.environ["SYNBIOCROW_BIOPKS_ROOT"]=str(BIOPKS_ROOT)
        os.environ["SYNBIOCROW_BIOPKS_RUNNER"]=str(REPO/"scripts/biopks_synbiocrow_bridge.py")
        run([sys.executable,str(REPO/"scripts/smoke_biopks_real.py")],cwd=REPO,timeout=600)
        print("BIOPKS BENCHMARK RUNTIME: READY")
    except Exception as exc:
        print("BIOPKS BENCHMARK RUNTIME: UNAVAILABLE",type(exc).__name__,str(exc))
else:
    print("BioPKS intentionally disabled.")


In [ ]:
# Restore the editable package after external installs, then capture the actual readiness matrix.
status(62,"restore SynBioCrow and inspect backend readiness")
run([sys.executable,"-m","pip","install","-q","-e",str(REPO)],timeout=600)
if str(REPO) not in sys.path: sys.path.insert(0,str(REPO))
importlib.invalidate_caches()

from synbiocrow import SynBioCrowEngine
engine=SynBioCrowEngine()
readiness=engine.backend_readiness()
print(json.dumps(readiness,indent=2,sort_keys=True))
(OUT/"backend_readiness.json").write_text(json.dumps(readiness,indent=2,sort_keys=True)+"\n")


In [ ]:
# Targeted harness regression before the scientific run.
status(68,"run benchmark harness tests")
run([sys.executable,"-m","pytest","-q","tests/test_benchmark_2_3.py"],cwd=REPO,timeout=600)


## Run / resume the 24-target benchmark
The output folder is fixed. If Colab disconnects, run the notebook again; completed `targets/<target_id>.json` files are reused automatically.


In [ ]:
status(72,"run/resume 24-target manuscript benchmark with live streaming")
cmd=[
    sys.executable,str(REPO/"scripts/run_benchmark_2_3.py"),
    "--panel",str(PANEL),
    "--output-dir",str(OUT),
    "--max-route-steps","8",
    "--max-routes","100",
]
print("$"," ".join(map(str,cmd)),flush=True)
proc=subprocess.Popen(cmd,cwd=REPO,text=True,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,bufsize=1)
last=time.time(); start=time.time()
for line in proc.stdout:
    print(line.rstrip(),flush=True)
    last=time.time()
rc=proc.wait()
if rc!=0:
    raise subprocess.CalledProcessError(rc,cmd)
status(94,"benchmark run complete")


In [ ]:
# Freeze a compact evidence bundle in Drive.
status(96,"build benchmark evidence bundle")
summary=json.loads((OUT/"benchmark_summary.json").read_text())
manifest=json.loads((OUT/"benchmark_manifest.json").read_text())
bundle=OUT/"SynBioCrow_2_3_MANUSCRIPT_BENCHMARK_V1_RESULTS.zip"
with zipfile.ZipFile(bundle,"w",zipfile.ZIP_DEFLATED) as z:
    for name in ["benchmark_manifest.json","benchmark_summary.json","benchmark_long.csv","backend_readiness.json"]:
        p=OUT/name
        if p.is_file(): z.write(p,p.name)
    for p in sorted((OUT/"targets").glob("*.json")):
        z.write(p,"targets/"+p.name)
def sha256(p):
    h=hashlib.sha256()
    with open(p,"rb") as f:
        for c in iter(lambda:f.read(1024*1024),b""): h.update(c)
    return h.hexdigest()
print(json.dumps(summary,indent=2,sort_keys=True))
print("manifest:",OUT/"benchmark_manifest.json")
print("bundle:",bundle)
print("bundle sha256:",sha256(bundle))
print("panel sha256:",manifest["panel_sha256"])
status(100,"MANUSCRIPT BENCHMARK V1 COMPLETE")
print("UPLOAD/REVIEW FILE:",bundle)
